## Colab on CUDA programming

This Jupyter nodebook is to demonstrate how to use Colab to do CUDA programming. 

1. Open your Google Drive on Chrome. Upload the provided tensor_cuda folder to your Google drive. It contains a Jupyter notebook cuda_colab.ipynb
2. Open cuda_colab.ipynb by double clicking it. 
3. From Runtime -> Change runtime type, select T4 GPU, save
4. Run the following commands to ensure the output is like provided. 

In [ ]:
!echo "Hello from bash"
!cat /etc/os-release
!uname -a
!pwd
!ls -l
!lscpu
!nvidia-smi

Note that colab notebook runs on Ubuntu server, and binds with a session. The session is closed when notebook is closed. 
!command is run a Linux command with Jypyter's Python cell. 
!pwd shows the path of current directory of the jupyter nodebook, e.g. /content, binding with current session. A new content directory will be created everytime a notebook is opened, removed after the session is closed.  
!ls -l lists files and folders under the current working directory
!nvidia-smi shows information of NVIDIA devices

In [ ]:
!nvcc --version

The following cell writes a program to a file in Python cell. The file is written under the current working directory content. The file will be removed after the notebook is closed.  

In [ ]:
%%writefile vector_add.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = threadIdx.x + blockDim.x * blockIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 100;
    size_t size = n * sizeof(int);

    // Allocate host memory
    int *h_a = (int *)malloc(size);
    int *h_b = (int *)malloc(size);
    int *h_c = (int *)malloc(size);

    // Initialize vectors
    for (int i = 0; i < n; i++) {
        h_a[i] = i;
        h_b[i] = 2 * i;
    }

    // Allocate device memory
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // Copy vectors from host to device
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // Launch kernel
    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);

    // Copy result back to host
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first 10 results
    for (int i = 0; i < 10; i++) {
        printf("%d + %d = %d\n", h_a[i], h_b[i], h_c[i]);
    }

    // Free memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc -arch=sm_75 vector_add.cu -o vector_add

In [ ]:
!./vector_add

0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27


You can run the program using nvprof to see the source usage.

In [ ]:
!nvprof ./vector_add

Save the stdout to a file

In [ ]:
!./vector_add > result.txt

In [ ]:
!ls

The following download the result.txt file to local machine.

In [ ]:
from google.colab import files
files.download('result.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

You can mount Google drive to the notebook's current session file system. There are thres ways:

1. By cell command: !drive.mount('/content/drive')
2. Click the file folder icon, then click the Google drive icon on the top.   
3. By the following Python code 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

After mouting, you copy files to Google's folder like the following.

In [ ]:
!cp result.txt /content/drive/MyDrive/tensor_cuda/result.txt

You can change session current working directory to Google Drive directory.

In [ ]:
import os

# Print the current working directory
print("Current directory:", os.getcwd())

# Create a new directory
new_dir = "drive/MyDrive/tensor_cuda"
# os.makedirs(new_dir, exist_ok=True)
# print(f"Created directory: {new_dir}")

# Change the working directory
os.chdir(new_dir)

# Print the new working directory
print("New current directory:", os.getcwd())

In [ ]:
!pwd
!ls -l

Then writing file and running program within the notebook will be in your Google's drive directory. 

In [ ]:
%%writefile dot_product.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void dotProduct(const float* A, const float* B, float* C, int N) {
    __shared__ float cache[1024];
    int i = threadIdx.x + blockDim.x * blockIdx.x;

    if(i < N) {
        cache[threadIdx.x] = A[i] * B[i];
    }

    __syncthreads();

    int i_half = blockDim.x / 2;
    while(i_half > 0) {
        if(threadIdx.x < i_half) {
            cache[threadIdx.x] += cache[threadIdx.x + i_half];
        }
        __syncthreads();
        i_half /= 2;
    }

    if(threadIdx.x == 0) {
        C[blockIdx.x] = cache[0];
    }
}

int main() {
    const int N = 1024;
    const int size = N * sizeof(float);

    float h_A[N], h_B[N];
    for (int i = 0; i < N; ++i) {
        h_A[i] = i;
        h_B[i] = i;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, sizeof(float));

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dotProduct<<<1, N>>>(d_A, d_B, d_C, N);

    float result;
    cudaMemcpy(&result, d_C, sizeof(float), cudaMemcpyDeviceToHost);

    printf("Dot Product: %.1f\n", result);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing dot_product.cu


In [ ]:
!nvcc -arch=sm_75 dot_product.cu -o dot_product

In [ ]:
!./dot_product > dot_product_result.txt

It is more convenient to use terminal command to write, build and run programs. Colab provide terminal to the Ubuntu system. You open the terminal by clicking the terminal icon at the bottom-left. Then cd to the google drive directory tensor_cuda, and use commmand to build and run the required cpp and cu program. 

Once you get the CUDA environment working, next you can focus on writing required code of tensor programs. You can do it in three ways:
1. Write all source code in VS code, and then upload to Google Drive and build test there. Copilot can be used to help the development. 
2. Write the code in jupyter notebook and test it within the notebook. Gemini can be used to help the development.
3. Write the code in old style using simple text editor vi under the termial.  